# Olive Fly Detection System

This notebook combines the training and classification components for the olive fly detection project.

## Part 1: Training the Neural Network Classifier

In [59]:
import os
import glob
import cv2
import numpy as np
from skimage.measure import label
from sklearn.neural_network import MLPClassifier

In [60]:
# Color-space foreground segmentation.
# Trap background is saturated yellow; insects/debris are dark and non-yellow.
# In CIELAB the yellow background has high b* and the foreground has low b*, so
# an Otsu split on the b* channel isolates the insect far more cleanly than a
# grayscale Otsu (which is fooled by shadows/lighting). Open+close clean specks.
def extract_foreground_mask(img, kernel_size=7):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    b_channel = lab[:, :, 2]
    _, bw = cv2.threshold(b_channel, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    labels = label(bw)
    if bw.sum() == 0:
        return np.zeros(bw.shape, np.uint8), labels
    largest = np.argmax(np.bincount(labels.flat, weights=bw.flat))
    return (labels == largest).astype(np.uint8), labels

# Image-level augmentation to balance the minority (olive_fly) class with
# varied positives instead of exact duplicates.
def augment(img, rng):
    out = img.copy()
    if rng.rand() < 0.5: out = cv2.flip(out, 1)
    if rng.rand() < 0.5: out = cv2.flip(out, 0)
    M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2),
                                rng.uniform(-30, 30), rng.uniform(0.9, 1.1))
    out = cv2.warpAffine(out, M, (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REFLECT)
    out = np.clip(rng.uniform(0.85, 1.15) * out.astype(np.float32) + rng.uniform(-25, 25),
                  0, 255).astype(np.uint8)
    return out

In [61]:
# 13 features: shape (area_frac, aspect, solidity, circularity, eccentricity,
# hu1, n_components) + colour (mean H/S/V, Lab a*/b* under mask, fg-bg contrast)
def compute_features(image):
    H, W = image.shape[:2]
    mask, labels = extract_foreground_mask(image)
    area = float(mask.sum())
    area_frac = area / (H * W)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    aspect = solidity = circ = ecc = hu1 = 0.0
    if contours:
        c = max(contours, key=cv2.contourArea)
        ca = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        aspect = float(w) / h if h > 0 else 1.0
        hull = cv2.contourArea(cv2.convexHull(c))
        solidity = ca / hull if hull > 0 else 0.0
        perim = cv2.arcLength(c, True)
        circ = min(4 * np.pi * ca / (perim * perim), 1.5) if perim > 0 else 0.0
        if len(c) >= 5:
            (_, _), (MA, ma), _ = cv2.fitEllipse(c)
            ecc = min(max(MA, ma) / min(MA, ma), 5.0) if min(MA, ma) > 0 else 0.0
        hu = cv2.HuMoments(cv2.moments(c)).flatten()
        hu1 = float(-np.sign(hu[0]) * np.log10(abs(hu[0]) + 1e-30))

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    m = mask.astype(bool)
    if m.any():
        hue = float(hsv[:, :, 0][m].mean())
        sat = float(hsv[:, :, 1][m].mean())
        val = float(hsv[:, :, 2][m].mean())
        lab_a = float(lab[:, :, 1][m].mean())
        lab_b = float(lab[:, :, 2][m].mean())
        bg = gray[~m].mean() if (~m).any() else gray[m].mean()
        contrast = float(bg - gray[m].mean())
    else:
        hue = sat = val = lab_a = lab_b = contrast = 0.0

    counts = np.bincount(labels.flat)
    if len(counts): counts[0] = 0
    ncomp = float((counts > 0.1 * area).sum()) if area > 0 else 0.0

    return np.array([area_frac, aspect, solidity, circ, ecc,
                     hue, sat, val, lab_a, lab_b, ncomp, contrast, hu1])

In [62]:
# Train the model: load images, balance the minority class with augmentation,
# fit a tiny MLP (13 -> 10 -> 1), and tune the decision threshold for best F1.
rng = np.random.RandomState(42)

print("Reading data...")
pos = [cv2.imread(f) for f in (glob.glob("olive_fly/*.JPG") + glob.glob("olive_fly/*.jpg"))]
neg = [cv2.imread(f) for f in (glob.glob("not_olive_fly/*.JPG") + glob.glob("not_olive_fly/*.jpg"))]
pos = [i for i in pos if i is not None]
neg = [i for i in neg if i is not None]
print(f"Class counts before balancing: olive_fly={len(pos)}, not_olive_fly={len(neg)}")

# Balance via augmentation of the minority class (varied synthetic positives).
aug = [augment(pos[rng.randint(len(pos))], rng) for _ in range(len(neg) - len(pos))]
feats = lambda imgs: np.array([compute_features(i) for i in imgs])
X = np.vstack([feats(pos), feats(aug), feats(neg)])
y = np.array([1] * (len(pos) + len(aug)) + [0] * len(neg))
print(f"Class counts after balancing: olive_fly={int((y==1).sum())}, not_olive_fly={int((y==0).sum())}")

# Feature standardization
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std[X_std == 0] = 1e-8
X_scaled = (X - X_mean) / X_std

mlp = MLPClassifier(hidden_layer_sizes=(10,), activation='relu', max_iter=8000, random_state=42)
mlp.fit(X_scaled, y)

# Tune the decision threshold on the real (un-augmented) images by F1.
Xreal = np.vstack([feats(pos), feats(neg)])
yreal = np.array([1] * len(pos) + [0] * len(neg))
proba = mlp.predict_proba((Xreal - X_mean) / X_std)[:, 1]
def f1_at(t):
    pred = proba >= t
    tp = int((pred & (yreal == 1)).sum()); fp = int((pred & (yreal == 0)).sum()); fn = int((~pred & (yreal == 1)).sum())
    if tp == 0: return 0.0
    p = tp / (tp + fp); r = tp / (tp + fn); return 2 * p * r / (p + r)
THRESHOLD = float(max(np.linspace(0.1, 0.9, 81), key=f1_at))
print(f"Tuned THRESHOLD = {THRESHOLD:.2f}")

print(f"X_MEAN = np.array({list(X_mean)})")
print(f"X_STD  = np.array({list(X_std)})")
print(f"W1 = np.array({mlp.coefs_[0].tolist()})")
print(f"B1 = np.array({mlp.intercepts_[0].tolist()})")
print(f"W2 = np.array({mlp.coefs_[1].tolist()})")
print(f"B2 = np.array({mlp.intercepts_[1].tolist()})")

Reading data...
Class counts before balancing: olive_fly=315, not_olive_fly=2035
Class counts after balancing: olive_fly=2035, not_olive_fly=2035
Tuned THRESHOLD = 0.69
X_MEAN = np.array([np.float64(0.08439716722069665), np.float64(1.0984064648035687), np.float64(0.7884098318154954), np.float64(0.4748295712131367), np.float64(1.9709271381403461), np.float64(36.8436394398803), np.float64(123.53874386339203), np.float64(91.4220614122015), np.float64(130.7495848177247), np.float64(143.09634910365688), np.float64(2.889189189189189), np.float64(46.99496336111946), np.float64(0.596722339365745)])
X_STD  = np.array([np.float64(0.07435869140689294), np.float64(0.5454724231463537), np.float64(0.13334792405954676), np.float64(0.1759800297348984), np.float64(0.7757921089104287), np.float64(28.63303994048818), np.float64(50.89852176812405), np.float64(47.61090497284364), np.float64(4.546115880094244), np.float64(12.562426301498357), np.float64(2.231504617091184), np.float64(31.186510703967752), np

## Part 2: Olive Fly Classification and Detection

In [63]:
import glob
import cv2
import numpy as np
from skimage.measure import label

In [64]:
MODEL_PARAMS = {
    'X_MEAN': X_mean,
    'X_STD': X_std,
    'W1': mlp.coefs_[0],
    'B1': mlp.intercepts_[0],
    'W2': mlp.coefs_[1],
    'B2': mlp.intercepts_[1]
}

X_MEAN = MODEL_PARAMS['X_MEAN']
X_STD = MODEL_PARAMS['X_STD']

W1 = MODEL_PARAMS['W1']
B1 = MODEL_PARAMS['B1']

W2 = MODEL_PARAMS['W2']
B2 = MODEL_PARAMS['B2']


In [65]:
# Color-space foreground segmentation.
# Trap background is saturated yellow; insects/debris are dark and non-yellow.
# In CIELAB the yellow background has high b* and the foreground has low b*, so
# an Otsu split on the b* channel isolates the insect far more cleanly than a
# grayscale Otsu (which is fooled by shadows/lighting). Open+close clean specks.
def extract_foreground_mask(img, kernel_size=7):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    b_channel = lab[:, :, 2]
    _, bw = cv2.threshold(b_channel, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)
    labels = label(bw)
    if bw.sum() == 0:
        return np.zeros(bw.shape, np.uint8), labels
    largest = np.argmax(np.bincount(labels.flat, weights=bw.flat))
    return (labels == largest).astype(np.uint8), labels

# 13 features: shape (area_frac, aspect, solidity, circularity, eccentricity,
# hu1, n_components) + colour (mean H/S/V, Lab a*/b* under mask, fg-bg contrast)
def compute_features(image):
    H, W = image.shape[:2]
    mask, labels = extract_foreground_mask(image)
    area = float(mask.sum())
    area_frac = area / (H * W)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    aspect = solidity = circ = ecc = hu1 = 0.0
    if contours:
        c = max(contours, key=cv2.contourArea)
        ca = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        aspect = float(w) / h if h > 0 else 1.0
        hull = cv2.contourArea(cv2.convexHull(c))
        solidity = ca / hull if hull > 0 else 0.0
        perim = cv2.arcLength(c, True)
        circ = min(4 * np.pi * ca / (perim * perim), 1.5) if perim > 0 else 0.0
        if len(c) >= 5:
            (_, _), (MA, ma), _ = cv2.fitEllipse(c)
            ecc = min(max(MA, ma) / min(MA, ma), 5.0) if min(MA, ma) > 0 else 0.0
        hu = cv2.HuMoments(cv2.moments(c)).flatten()
        hu1 = float(-np.sign(hu[0]) * np.log10(abs(hu[0]) + 1e-30))

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    m = mask.astype(bool)
    if m.any():
        hue = float(hsv[:, :, 0][m].mean())
        sat = float(hsv[:, :, 1][m].mean())
        val = float(hsv[:, :, 2][m].mean())
        lab_a = float(lab[:, :, 1][m].mean())
        lab_b = float(lab[:, :, 2][m].mean())
        bg = gray[~m].mean() if (~m).any() else gray[m].mean()
        contrast = float(bg - gray[m].mean())
    else:
        hue = sat = val = lab_a = lab_b = contrast = 0.0

    counts = np.bincount(labels.flat)
    if len(counts): counts[0] = 0
    ncomp = float((counts > 0.1 * area).sum()) if area > 0 else 0.0

    return np.array([area_frac, aspect, solidity, circ, ecc,
                     hue, sat, val, lab_a, lab_b, ncomp, contrast, hu1])

In [66]:

def detect_olive_fly(image) -> bool:

    try:
        # 1. Extract and standardize physical attributes
        raw_features = compute_features(image)
        scaled_features = (raw_features - X_MEAN) / X_STD

        # 2. Hidden Layer Propagation: Z1 = A1*W1 + B1
        z1 = np.dot(scaled_features, W1) + B1
        a2 = np.maximum(0, z1)  # ReLU non-linear activation function

        # 3. Output Layer Propagation: Z2 = A2*W2 + B2
        z2 = np.dot(a2, W2) + B2

        # Sigmoid activation function mapping to classification boundaries
        probability = 1 / (1 + np.exp(-np.clip(z2[0], -500, 500)))

        return bool(probability >= THRESHOLD)

    except Exception:
        return False

### Automated Evaluation

In [67]:
def evaluate_classifier(directory):
    TP, TN, FP, FN = 0, 0, 0, 0
    
    for filename in glob.glob(str(directory)+"/**/*.JPG", recursive=True):
        img = cv2.imread(filename)
        if "not_olive_fly" in filename:
            olive_fly = False
        elif "olive_fly" in filename:
            olive_fly = True
        else:
            print(f"{filename} not labeled.")
            continue

        detection_result = detect_olive_fly(img)

        if olive_fly and detection_result:
            TP += 1
        elif olive_fly and not detection_result:
            FN += 1
        elif not olive_fly and detection_result:
            FP += 1
        else:
            TN += 1
            
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    if (TP + FP) > 0 and (TP + FN) > 0:
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        accuracy = (TP + TN) / (TP + TN + FP + FN)
        print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, Accuracy: {accuracy:.4f}")

In [68]:
print("Evaluating on training data...\n")
evaluate_classifier(".")

Evaluating on training data...

TP: 221, TN: 1888, FP: 147, FN: 94
Precision: 0.6005, Recall: 0.7016, Accuracy: 0.8974


In [69]:
test_image_path = "olive_fly/IMG_0597 1 referencia.JPG"


if os.path.exists(test_image_path):
    test_img = cv2.imread(test_image_path)
    if test_img is not None:
        result = detect_olive_fly(test_img)
        print(f"Image: {test_image_path}")
        print(f"Detection Result: {'Olive Fly Detected' if result else 'No Olive Fly'}")
    else:
        print(f"Could not read image: {test_image_path}")
else:
    print(f"File not found: {test_image_path}")

Image: olive_fly/IMG_0597 1 referencia.JPG
Detection Result: Olive Fly Detected
